# Human Spotcheck — 100 Sample (Word Doc Format)

**Why Word docs instead of Excel?**
Excel cells truncate at 32,767 characters. Some Gemini ReAct outputs are 44,000+ chars, so Excel was silently cutting them off. Word documents have no such limit, and they're easier for raters to read.

**Output structure:**
```
Human-Raters/
├── Opeyemi/
│   ├── 001_Abuse013_Claude_LeastToMost.docx
│   ├── 002_...
│   └── Opeyemi-scores.xlsx        ← rater fills H1-H6 here
└── Derrick/
    ├── 001_Abuse013_Claude_LeastToMost.docx   (identical to Opeyemi's)
    ├── 002_...
    └── Derrick-scores.xlsx        ← rater fills H1-H6 here
```

Both raters get the SAME 100 Word docs (same row_id ordering), but separate scoring spreadsheets.

**Stage A** generates the folders and files.
**Stage B** loads the filled scoring spreadsheets and computes agreement.

---
## STAGE A — Generate sample + Word docs + scoring sheets

In [1]:
# A1: imports + paths
import os
import json
import re
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# Word doc generation
try:
    from docx import Document
    from docx.shared import Pt, RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    raise SystemExit('python-docx not installed. Run:  pip install python-docx')

# ============================================================
# CONFIGURATION
# ============================================================
OUTPUT_DIR = r'C:\Opeyemi\PROMPTS\EVALUATION'
RATERS_DIR = r'C:\Opeyemi\PROMPTS\Human-Raters'
os.makedirs(RATERS_DIR, exist_ok=True)

# Per-rater folders
OPEYEMI_DIR = os.path.join(RATERS_DIR, 'Opeyemi')
DERRICK_DIR = os.path.join(RATERS_DIR, 'Derrick')
os.makedirs(OPEYEMI_DIR, exist_ok=True)
os.makedirs(DERRICK_DIR, exist_ok=True)

FULL_LABELED_PATH  = os.path.join(OUTPUT_DIR, 'full_labeled_dataset_subset.csv')
AGREEMENT_PATH     = os.path.join(OUTPUT_DIR, 'human_panel_agreement.json')
OPEYEMI_SCORES     = os.path.join(OPEYEMI_DIR, 'Opeyemi-scores.xlsx')
DERRICK_SCORES     = os.path.join(DERRICK_DIR, 'Derrick-scores.xlsx')
OPEYEMI_FILLED     = os.path.join(OPEYEMI_DIR, 'Opeyemi-scores_FILLED.xlsx')
DERRICK_FILLED     = os.path.join(DERRICK_DIR, 'Derrick-scores_FILLED.xlsx')

HCOLS       = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
N_SAMPLE    = 100
RANDOM_SEED = 42

print(f'Output dir   : {OUTPUT_DIR}')
print(f'Opeyemi dir  : {OPEYEMI_DIR}')
print(f'Derrick dir  : {DERRICK_DIR}')
print(f'Sample size  : {N_SAMPLE}')

Output dir   : C:\Opeyemi\PROMPTS\EVALUATION
Opeyemi dir  : C:\Opeyemi\PROMPTS\Human-Raters\Opeyemi
Derrick dir  : C:\Opeyemi\PROMPTS\Human-Raters\Derrick
Sample size  : 100


In [2]:
# A2: load full labeled dataset and stratified-sample N_SAMPLE rows
df_full = pd.read_csv(FULL_LABELED_PATH)
print(f'Loaded full labeled dataset: {len(df_full):,} rows')
print(f'\nPer (model, technique):')
print(df_full.groupby(['model', 'technique']).size().unstack(fill_value=0))

# Stratified across model × technique (12 cells)
groups = df_full.groupby(['model', 'technique'])
n_groups = len(groups)
base_per_group = N_SAMPLE // n_groups
remainder = N_SAMPLE - base_per_group * n_groups

print(f'\nSampling: {base_per_group} per (model, technique) cell + {remainder} extras')

rng = np.random.RandomState(RANDOM_SEED)
sample_dfs = []
extras_pool = []

for (model, tech), g in groups:
    take = min(base_per_group, len(g))
    picked = g.sample(n=take, random_state=RANDOM_SEED)
    sample_dfs.append(picked)
    leftover = g.drop(picked.index)
    extras_pool.append(leftover)

df_sample = pd.concat(sample_dfs, ignore_index=True)

# Top up to exactly N_SAMPLE
if remainder > 0 and len(df_sample) == base_per_group * n_groups:
    extras_df = pd.concat(extras_pool, ignore_index=True)
    if len(extras_df) >= remainder:
        topup = extras_df.sample(n=remainder, random_state=RANDOM_SEED + 1)
        df_sample = pd.concat([df_sample, topup], ignore_index=True)
elif len(df_sample) < N_SAMPLE:
    extras_df = pd.concat(extras_pool, ignore_index=True)
    need = N_SAMPLE - len(df_sample)
    if len(extras_df) >= need:
        topup = extras_df.sample(n=need, random_state=RANDOM_SEED)
        df_sample = pd.concat([df_sample, topup], ignore_index=True)

df_sample = df_sample.reset_index(drop=True)
df_sample['row_id'] = [f'{i+1:03d}' for i in range(len(df_sample))]

print(f'\nFinal sample: {len(df_sample)} rows')
print(f'\nSample distribution per (model, technique):')
print(df_sample.groupby(['model', 'technique']).size().unstack(fill_value=0))
print(f'\nSample distribution per crime_type:')
print(df_sample.groupby('crime_type').size())

Loaded full labeled dataset: 616 rows

Per (model, technique):
technique  Least-to-Most  ReAct  Sequential  Zero-Shot
model                                                 
Claude                55     55          55         55
GPT                   55     55          55         55
Gemini                16     55          50         55

Sampling: 8 per (model, technique) cell + 4 extras

Final sample: 100 rows

Sample distribution per (model, technique):
technique  Least-to-Most  ReAct  Sequential  Zero-Shot
model                                                 
Claude                 9      8           8          9
GPT                    8      8           8          8
Gemini                 8      8          10          8

Sample distribution per crime_type:
crime_type
Abuse             2
Assault          13
Burglary         14
Explosion        12
Fighting          1
RoadAccidents    13
Robbery          21
Shooting          1
Shoplifting      10
Stealing         13
dtype: int64


In [3]:
# A3: helper — generate one Word doc per row
def safe_filename(s):
    """Make string safe for Windows filenames."""
    return re.sub(r'[\\/:*?"<>|]', '_', str(s))

def make_rater_docx(out_path, row):
    """Create one Word doc with full ground truth + full model output + scoring checklist."""
    doc = Document()

    # Default font sizing
    style = doc.styles['Normal']
    style.font.name = 'Calibri'
    style.font.size = Pt(11)

    # ===== Header / metadata =====
    h = doc.add_heading(f'Spotcheck Row {row["row_id"]}', level=1)
    h.alignment = WD_ALIGN_PARAGRAPH.LEFT

    p = doc.add_paragraph()
    p.add_run('Video: ').bold = True
    p.add_run(str(row['video']))

    p = doc.add_paragraph()
    p.add_run('Crime type: ').bold = True
    p.add_run(str(row['crime_type']))

    p = doc.add_paragraph()
    p.add_run('Model / Technique: ').bold = True
    p.add_run(f'{row["model"]} / {row["technique"]}')

    doc.add_paragraph()   # spacer

    # ===== Ground truth =====
    doc.add_heading('Ground Truth', level=2)
    doc.add_paragraph(str(row['ground_truth']))

    # ===== Model output =====
    doc.add_heading('Model Output', level=2)
    # Word handles unlimited length — split paragraphs on \n\n for readability
    text = str(row['model_output'])
    for chunk in text.split('\n\n'):
        if chunk.strip():
            doc.add_paragraph(chunk.strip())

    # ===== Scoring instructions =====
    doc.add_page_break()
    doc.add_heading('Hallucination Scoring Instructions', level=2)

    instructions = (
        'Compare the MODEL OUTPUT against the GROUND TRUTH above. For each '
        'hallucination type below, decide whether it is PRESENT (1) or ABSENT (0) '
        'in the model output. Then enter your scores in the file '
        '"<Your-Name>-scores.xlsx" in the row matching this Row ID.'
    )
    doc.add_paragraph(instructions)

    descriptions = [
        ('H1  SCENE_FABRICATION',       'Invents setting details not in ground truth (wrong location, objects, environment).'),
        ('H2  CRIME_MISCLASSIFICATION', 'Identifies the wrong crime type (e.g. says Robbery when it is Assault).'),
        ('H3  CRIME_MISSED',            'Fails to mention the primary crime that is clearly described in ground truth.'),
        ('H4  SEVERITY_MINIMIZATION',   'Describes the crime as less serious than the ground truth indicates.'),
        ('H5  ENTITY_FABRICATION',      'Invents people, vehicles, or objects not present in ground truth.'),
        ('H6  PHANTOM_ACTORS',          'Adds perpetrators or victims not mentioned in ground truth.'),
    ]

    for label, desc in descriptions:
        p = doc.add_paragraph()
        p.add_run(label).bold = True
        p.add_run(' — ' + desc)

    doc.add_paragraph()
    p = doc.add_paragraph()
    p.add_run('Reminder: ').bold = True
    p.add_run(
        'Score independently. Do NOT discuss with the other rater until both '
        'spreadsheets are submitted.'
    )

    doc.save(out_path)

print('docx generator ready.')

docx generator ready.


In [4]:
# A4: generate all 100 Word docs into BOTH rater folders
from tqdm.auto import tqdm

print(f'Generating {len(df_sample)} Word docs per rater...')

for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc='Word docs'):
    fname_base = (
        f'{row["row_id"]}_'
        f'{safe_filename(row["video"])}_'
        f'{safe_filename(row["model"])}_'
        f'{safe_filename(row["technique"])}.docx'
    )
    op_path = os.path.join(OPEYEMI_DIR, fname_base)
    dk_path = os.path.join(DERRICK_DIR, fname_base)
    make_rater_docx(op_path, row)
    make_rater_docx(dk_path, row)

print(f'\nWrote {len(df_sample)} docs into each of:')
print(f'  {OPEYEMI_DIR}')
print(f'  {DERRICK_DIR}')

Generating 100 Word docs per rater...


Word docs:   0%|          | 0/100 [00:00<?, ?it/s]


Wrote 100 docs into each of:
  C:\Opeyemi\PROMPTS\Human-Raters\Opeyemi
  C:\Opeyemi\PROMPTS\Human-Raters\Derrick


In [5]:
# A5: build the scoring spreadsheets — one per rater
# These are SMALL — just the row identifiers and the H1-H6 columns to fill in.
# Panel labels are kept on Derrick's sheet for direct comparison in Stage B.

scores_cols = ['row_id', 'video', 'crime_type', 'model', 'technique',
               'human_H1', 'human_H2', 'human_H3', 'human_H4', 'human_H5', 'human_H6',
               'human_notes']

df_scores = df_sample.copy()
for h in HCOLS:
    df_scores[f'human_{h}'] = ''   # blank for the rater to fill in
df_scores['human_notes'] = ''

# Opeyemi's scoring file — panel labels included for direct comparison in Stage B
df_op_out = df_scores[scores_cols].copy()
for h in HCOLS:
    df_op_out[f'panel_{h}'] = df_sample[h].astype(int).values
df_op_out.to_excel(OPEYEMI_SCORES, index=False)
print(f'Saved: {OPEYEMI_SCORES}')

# Derrick's scoring file — same panel labels for symmetric comparison
df_dk_out = df_scores[scores_cols].copy()
for h in HCOLS:
    df_dk_out[f'panel_{h}'] = df_sample[h].astype(int).values
df_dk_out.to_excel(DERRICK_SCORES, index=False)
print(f'Saved: {DERRICK_SCORES}')

print('\n=== Rater package summary ===')
print(f'Each rater receives:')
print(f'  - 100 Word docs ({len(df_sample)} files)')
print(f'  - 1 scoring spreadsheet (Opeyemi-scores.xlsx / Derrick-scores.xlsx)')
print()
print('Workflow for each rater:')
print('  1. Open each Word doc to read the ground truth and model output')
print('  2. Open the scoring spreadsheet')
print('  3. For each row_id, enter 0 or 1 for human_H1 through human_H6')
print('  4. Optionally add a note in human_notes')
print('  5. Save the spreadsheet as <Name>-scores_FILLED.xlsx and return it')

Saved: C:\Opeyemi\PROMPTS\Human-Raters\Opeyemi\Opeyemi-scores.xlsx
Saved: C:\Opeyemi\PROMPTS\Human-Raters\Derrick\Derrick-scores.xlsx

=== Rater package summary ===
Each rater receives:
  - 100 Word docs (100 files)
  - 1 scoring spreadsheet (Opeyemi-scores.xlsx / Derrick-scores.xlsx)

Workflow for each rater:
  1. Open each Word doc to read the ground truth and model output
  2. Open the scoring spreadsheet
  3. For each row_id, enter 0 or 1 for human_H1 through human_H6
  4. Optionally add a note in human_notes
  5. Save the spreadsheet as <Name>-scores_FILLED.xlsx and return it


---
## STAGE B — Analyze filled scoring sheets

Run after Opeyemi and Derrick return their FILLED spreadsheets.

Expected files:
- `Human-Raters/Opeyemi/Opeyemi-scores_FILLED.xlsx`
- `Human-Raters/Derrick/Derrick-scores_FILLED.xlsx`

In [6]:
# B1: load both filled scoring sheets, validate, and compute ANY columns
assert os.path.exists(OPEYEMI_FILLED), f'Missing: {OPEYEMI_FILLED}'
assert os.path.exists(DERRICK_FILLED), f'Missing: {DERRICK_FILLED}'

df_op = pd.read_excel(OPEYEMI_FILLED)
df_dk = pd.read_excel(DERRICK_FILLED)

# Sort both by row_id to ensure alignment
df_op = df_op.sort_values('row_id').reset_index(drop=True)
df_dk = df_dk.sort_values('row_id').reset_index(drop=True)

# Verify alignment
assert (df_op['row_id'] == df_dk['row_id']).all(), 'row_id mismatch between rater files!'
assert (df_op['video']  == df_dk['video']).all(),  'video mismatch between rater files!'
n_rows = len(df_op)
print(f'Loaded {n_rows} rows from each rater (aligned by row_id).')

# Coerce blanks to 0 / ints
for h in HCOLS:
    df_op[f'human_{h}'] = pd.to_numeric(df_op[f'human_{h}'], errors='coerce').fillna(0).astype(int)
    df_dk[f'human_{h}'] = pd.to_numeric(df_dk[f'human_{h}'], errors='coerce').fillna(0).astype(int)
    df_dk[f'panel_{h}'] = pd.to_numeric(df_dk[f'panel_{h}'], errors='coerce').fillna(0).astype(int)

# ANY columns
any_op = (df_op[[f'human_{h}' for h in HCOLS]].sum(axis=1) > 0).astype(int)
any_dk = (df_dk[[f'human_{h}' for h in HCOLS]].sum(axis=1) > 0).astype(int)
any_pa = (df_dk[[f'panel_{h}'  for h in HCOLS]].sum(axis=1) > 0).astype(int)

print('\n=== Rating Summary (count of 1s) ===')
print(f'{"H-type":<8} {"n+_Opeyemi":>12} {"n+_Derrick":>12} {"n+_Panel":>10}')
print('-' * 47)
for h in HCOLS:
    print(f'{h:<8} {df_op[f"human_{h}"].sum():>12} '
          f'{df_dk[f"human_{h}"].sum():>12} '
          f'{df_dk[f"panel_{h}"].sum():>10}')
print(f'{"ANY":<8} {any_op.sum():>12} {any_dk.sum():>12} {any_pa.sum():>10}')

AssertionError: Missing: C:\Opeyemi\PROMPTS\Human-Raters\Opeyemi\Opeyemi-scores_FILLED.xlsx

In [ ]:
# B2: Inter-rater agreement (Opeyemi vs Derrick)
print('=== Inter-Rater Agreement: Opeyemi vs Derrick ===')
print(f'{"H-type":<8} {"Agree%":>8} {"Kappa":>8}')
print('-' * 28)

kappas_irr = []
irr_results = {}

for h in HCOLS:
    y_op = df_op[f'human_{h}'].values
    y_dk = df_dk[f'human_{h}'].values
    agree = (y_op == y_dk).mean() * 100
    try:
        k = cohen_kappa_score(y_op, y_dk)
    except Exception:
        k = float('nan')
    if not np.isnan(k):
        kappas_irr.append(k)
    irr_results[h] = {'agreement_pct': float(agree), 'kappa': float(k)}
    print(f'{h:<8} {agree:>7.1f}% {k:>8.3f}')

k_any_irr     = cohen_kappa_score(any_op.values, any_dk.values)
agree_any_irr = (any_op.values == any_dk.values).mean() * 100
irr_results['ANY']   = {'agreement_pct': float(agree_any_irr), 'kappa': float(k_any_irr)}
irr_results['macro'] = float(np.nanmean(kappas_irr))

print(f'{"ANY":<8} {agree_any_irr:>7.1f}% {k_any_irr:>8.3f}')
print(f'\nMacro kappa (H1-H6): {irr_results["macro"]:.3f}')
print(f'ANY kappa          : {k_any_irr:.3f}')

In [ ]:
# B3: Each human rater vs LLM panel
rater_panel_results = {}

for rater_name, df_rater, any_rater in [
    ('Opeyemi', df_op, any_op),
    ('Derrick', df_dk, any_dk)
]:
    print(f'=== {rater_name} vs Panel ===')
    print(f'{"H-type":<8} {"Agree%":>8} {"Kappa":>8}')
    print('-' * 28)

    kappas_r = []
    r_results = {}

    for h in HCOLS:
        y_rater = df_rater[f'human_{h}'].values
        y_panel = df_dk[f'panel_{h}'].values
        agree   = (y_rater == y_panel).mean() * 100
        try:
            k = cohen_kappa_score(y_rater, y_panel)
        except Exception:
            k = float('nan')
        if not np.isnan(k):
            kappas_r.append(k)
        r_results[h] = {'agreement_pct': float(agree), 'kappa': float(k)}
        print(f'{h:<8} {agree:>7.1f}% {k:>8.3f}')

    k_any_r     = cohen_kappa_score(any_rater.values, any_pa.values)
    agree_any_r = (any_rater.values == any_pa.values).mean() * 100
    r_results['ANY']   = {'agreement_pct': float(agree_any_r), 'kappa': float(k_any_r)}
    r_results['macro'] = float(np.nanmean(kappas_r))

    print(f'{"ANY":<8} {agree_any_r:>7.1f}% {k_any_r:>8.3f}')
    print(f'Macro kappa (H1-H6): {r_results["macro"]:.3f}\n')
    rater_panel_results[rater_name] = r_results

In [ ]:
# B4: Majority human vote vs panel (rows where both raters agree)
print('=== Majority Human Vote vs Panel ===')
print('(only rows where Opeyemi and Derrick agree)')
print(f'{"H-type":<8} {"n_agreed":>9} {"Agree%":>8} {"Kappa":>8}')
print('-' * 38)

kappas_maj = []
maj_results = {}

for h in HCOLS:
    y_op    = df_op[f'human_{h}'].values
    y_dk    = df_dk[f'human_{h}'].values
    y_panel = df_dk[f'panel_{h}'].values
    mask    = y_op == y_dk
    n_agree = mask.sum()
    if n_agree > 0:
        agree = (y_op[mask] == y_panel[mask]).mean() * 100
        try:
            k = cohen_kappa_score(y_op[mask], y_panel[mask])
        except Exception:
            k = float('nan')
    else:
        agree, k = 0.0, float('nan')
    if not np.isnan(k):
        kappas_maj.append(k)
    maj_results[h] = {
        'n_rows_agreed': int(n_agree),
        'agreement_pct': float(agree),
        'kappa': float(k)
    }
    print(f'{h:<8} {n_agree:>9} {agree:>7.1f}% {k:>8.3f}')

mask_any = any_op.values == any_dk.values
if mask_any.sum() > 0:
    k_any_maj     = cohen_kappa_score(any_op.values[mask_any], any_pa.values[mask_any])
    agree_any_maj = (any_op.values[mask_any] == any_pa.values[mask_any]).mean() * 100
else:
    k_any_maj, agree_any_maj = float('nan'), 0.0

maj_results['ANY']   = {
    'n_rows_agreed': int(mask_any.sum()),
    'agreement_pct': float(agree_any_maj),
    'kappa': float(k_any_maj)
}
maj_results['macro'] = float(np.nanmean(kappas_maj))

print(f'{"ANY":<8} {mask_any.sum():>9} {agree_any_maj:>7.1f}% {k_any_maj:>8.3f}')
print(f'\nMacro kappa (H1-H6): {maj_results["macro"]:.3f}')
print(f'ANY kappa          : {k_any_maj:.3f}')

In [ ]:
# B5: summary + save JSON
INTER_JUDGE_PATH = os.path.join(OUTPUT_DIR, 'inter_judge_agreement_subset.json')
panel_macro_range = '(see inter_judge_agreement_subset.json)'
if os.path.exists(INTER_JUDGE_PATH):
    with open(INTER_JUDGE_PATH) as f:
        ij = json.load(f)
    overall_kappas = [v['overall'] for k, v in ij.items() if 'overall' in v]
    if overall_kappas:
        panel_macro_range = f'{min(overall_kappas):.3f}-{max(overall_kappas):.3f}'

print('=' * 62)
print('SUMMARY: Human Validation Results')
print('=' * 62)
print(f'{"Comparison":<38} {"Macro kappa":>11} {"ANY kappa":>9}')
print('-' * 62)
print(f'{"Opeyemi vs Derrick (inter-rater)":<38} '
      f'{irr_results["macro"]:>11.3f} '
      f'{irr_results["ANY"]["kappa"]:>9.3f}')
print(f'{"Opeyemi vs Panel":<38} '
      f'{rater_panel_results["Opeyemi"]["macro"]:>11.3f} '
      f'{rater_panel_results["Opeyemi"]["ANY"]["kappa"]:>9.3f}')
print(f'{"Derrick vs Panel":<38} '
      f'{rater_panel_results["Derrick"]["macro"]:>11.3f} '
      f'{rater_panel_results["Derrick"]["ANY"]["kappa"]:>9.3f}')
print(f'{"Majority human vote vs Panel":<38} '
      f'{maj_results["macro"]:>11.3f} '
      f'{maj_results["ANY"]["kappa"]:>9.3f}')
print(f'{"LLM panel inter-judge overall kappa":<38} {panel_macro_range:>11}')
print(f'\nn spotcheck rows: {n_rows}')

agreement_results = {
    'n_rows': int(n_rows),
    'raters': ['Opeyemi', 'Derrick'],
    'inter_rater_agreement': irr_results,
    'opeyemi_vs_panel': rater_panel_results['Opeyemi'],
    'derrick_vs_panel': rater_panel_results['Derrick'],
    'majority_human_vs_panel': maj_results,
}

with open(AGREEMENT_PATH, 'w') as f:
    json.dump(agreement_results, f, indent=2)
print(f'\nSaved: {AGREEMENT_PATH}')